In [21]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


# **Exercice** 2

In [22]:
import math
import random

def is_prime(n):
    """Miller-Rabin primality test (fast for large numbers)."""
    if n < 2: return False
    if n in (2, 3): return True
    if n % 2 == 0: return False

    r, s = 0, n - 1
    while s % 2 == 0:
        r, s = r + 1, s // 2

    for _ in range(5):  # 5 rounds for strong accuracy
        a = random.randrange(2, n - 1)
        x = pow(a, s, n)
        if x in (1, n - 1):
            continue
        for _ in range(r - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False
    return True

def pollard_rho(n):
    """Pollard's Rho algorithm for fast factorization."""
    if n % 2 == 0:
        return 2
    x, y, c, g = random.randint(1, n-1), 2, random.randint(1, n-1), 1
    while g == 1:
        x = (pow(x, 2, n) + c) % n
        y = (pow(pow(y, 2, n) + c, 2, n) + c) % n
        g = math.gcd(abs(x - y), n)
    return g if g != n else None

def factorize(n):
    """Factorizes a large number into its prime factors."""
    factors, stack = [], [n]

    while stack:
        num = stack.pop()
        if is_prime(num):
            factors.append(num)
            continue

        factor = pollard_rho(num) or next(i for i in range(2, int(math.sqrt(num)) + 1) if num % i == 0)
        stack.extend([factor, num // factor])

    return sorted(factors)

# Example usage
n = 956331992007843552652604425031376690367  # Replace with your number
print(f"Factors of {n}: {factorize(n)}")


Factors of 956331992007843552652604425031376690367: [7746289204980135457, 123456789012345681631]


 I am implementing a program to factorize a large number (n) into its two prime factors. This factorization is crucial because it allows me to compute the private key (d), which is needed to decrypt an encrypted message in the RSA cryptosystem.

In [23]:
import gmpy2

def modinv(a, m):
    """Computes modular inverse using gmpy2."""
    return int(gmpy2.invert(a, m))

def decrypt(ciphertext, d, n):
    """RSA decryption with proper handling of custom letter encoding."""
    decrypted_numbers = [int(gmpy2.powmod(c, d, n)) for c in ciphertext]

    # Convert numbers to a string representation
    decrypted_text = "".join(str(num) for num in decrypted_numbers)

    # Map digits to letters based on the given scheme
    mapping = {"00": " ", **{str(i + 11): chr(65 + i) for i in range(26)}}  # Space + A-Z

    # Decode the message in chunks of 2 digits
    decoded_message = "".join(mapping.get(decrypted_text[i:i+2], "?") for i in range(0, len(decrypted_text), 2))

    return decoded_message

# --- Given RSA Public Key and Prime Factors ---
p = 7746289204980135457
q = 123456789012345681631
e = 12398737

# Compute n and φ(n)
n = p * q
phi = (p - 1) * (q - 1)

# Compute private exponent d
d = modinv(e, phi)

# --- User Input for Ciphertext ---
ciphertext_input = input("Enter ciphertext numbers separated by commas: ")
ciphertext = list(map(int, ciphertext_input.split(',')))

# --- Decrypt the Message ---
plaintext = decrypt(ciphertext, d, n)

# Print the decrypted message
print("Decrypted message:", plaintext)


Enter ciphertext numbers separated by commas:  427849968240759007228494978639775081809, 498308250136673589542748543030806629941, 925288105342943743271024837479707225255, 95024328800414254907217356783906225740
Decrypted message: THIS IS MY LETTER TO THE WORLD THAT NEVER WROTE TO ME EMILY DICKINSON


Using these factors, I compute Euler’s totient function φ(n) = (p - 1) * (q -1).
With φ(n), I calculate the modular inverse of the public exponent (e) to obtain the private key (d).
Finally, I use d to decrypt the ciphertext and retrieve the original message.



# Exercice 3.  

In [ ]:
def count_digits(n):
    return len(str(n))

# Example
num =  778316029000243099877931053342499782854299069475452815411870902004051418097561283720829540275065791136400699788752314702571941118322923543821748427207483967956024258259313108355124837548277681467200823621868472079516948529577547685807235469043078217078807400095317080158559720414127067108901243940363
print(count_digits(num))  # Output: 9


300


In [25]:
#!pip install gmpy2
#!pip install pycryptodome
#!pip install Crypto
import gmpy2
from Crypto.Util.number import getPrime, inverse

def generate_safe_prime(digits):
    """Generates a safe prime with at least 'digits' decimal digits."""
    bits = digits * 3.321928  # Convert decimal digits to bits
    bits = int(bits) + 1  # Ensure rounding up
    while True:
        p = getPrime(bits)  # Fast prime generation
        if gmpy2.is_prime((p - 1) // 2):  # Ensure safe prime condition
            return p

def generate_keypair(digits=400):
    """Generates RSA keypair with at least 'digits' in modulus."""
    print("Generating large safe primes...")
    p, q = generate_safe_prime(digits // 2), generate_safe_prime(digits // 2)

    n = p * q
    while len(str(n)) < digits:  # Ensure n has at least 400 digits
        p, q = generate_safe_prime(digits // 2), generate_safe_prime(digits // 2)
        n = p * q

    phi = (p - 1) * (q - 1)

    e = 65537  # Standard public exponent (fast, secure)
    d = inverse(e, phi)
    print("factorization  of p " , p)
    print("factorization  of q " , q)
    return (e, n), (d, n)

def encrypt(pk, plaintext):
    """Encrypt a message using RSA."""
    e, n = pk
    return [pow(ord(char), e, n) for char in plaintext]

def decrypt(pk, ciphertext):
    """Decrypt a message using RSA."""
    d, n = pk
    return ''.join(chr(pow(char, d, n)) for char in ciphertext)

# --- Main Program ---
if __name__ == '__main__':
    # Generate RSA keypair with at least 800-digit modulus
    print("Generating RSA keypair...")
    public, private = generate_keypair(800)
    print("Keypair generated.")

    print("Public key:", public)
    print("Private key:", private)

    # Encrypt and decrypt a message
    message = "Secret!"
    encrypted_msg = encrypt(public, message)
    print("Encrypted message:", encrypted_msg)

    decrypted_msg = decrypt(private, encrypted_msg)
    print("Decrypted message:", decrypted_msg)


Generating RSA keypair...
Generating large safe primes...
factorization  of p  9687070279716731473159553817082747138243592774542713654317157148831250759077547077835790161910985214302861116730234013460982907030000842331665875122926076979872034297435108889668328637886773466428748726766188307375671301186118275022589604547896424614903396046871484829555821191430820988170110999142485417843849793210942401569129455245736567352731770813732535656556052742659584302584765134690022032843
factorization  of q  8280121262157428189083663906786572923353242376824853436928848479738723321500389966253902693855456023751597788931649525604434114480222322627734265638358243579530135825381004087125471240573802038458378147509808564527030195266915911774067516143687920043274816593343290647531811224878641911159391689496568189463133224638475699802374096486981840530173400845209324842045975580312739475324208599581227841579
Keypair generated.
Public key: (65537, 80210116591095813539714990492996125575454563263359411596241

In [3]:
!pip install gmpy2
!pip install pycryptodome
!pip install Crypto
import gmpy2
from Crypto.Util.number import getPrime, inverse

def generate_safe_prime(digits):
    """Generates a safe prime with at least 'digits' decimal digits."""
    bits = digits * 3.321928  # Convert decimal digits to bits
    bits = int(bits) + 1  # Ensure rounding up
    while True:
        p = getPrime(bits)  # Fast prime generation
        if gmpy2.is_prime((p - 1) // 2):  # Ensure safe prime condition
            return p

def generate_keypair(digits=400):
    """Generates RSA keypair with at least 'digits' in modulus."""
    print("Generating large safe primes...")
    p, q = generate_safe_prime(digits // 2), generate_safe_prime(digits // 2)

    n = p * q
    while len(str(n)) < digits:  # Ensure n has at least 1000 digits
        p, q = generate_safe_prime(digits // 2), generate_safe_prime(digits // 2)
        n = p * q

    phi = (p - 1) * (q - 1)

    e = 65537  # Standard public exponent (fast, secure)
    d = inverse(e, phi)
    print("factorization  of p " , p)
    print("factorization  of q " , q)
    return (e, n), (d, n)

def encrypt(pk, plaintext):
    """Encrypt a message using RSA."""
    e, n = pk
    return [pow(ord(char), e, n) for char in plaintext]

def decrypt(pk, ciphertext):
    """Decrypt a message using RSA."""
    d, n = pk
    return ''.join(chr(pow(char, d, n)) for char in ciphertext)

# --- Main Program ---
if __name__ == '__main__':
    # Generate RSA keypair with at least 1000-digit modulus
    print("Generating RSA keypair...")
    public, private = generate_keypair(1000)
    print("Keypair generated.")

    print("Public key:", public)
    print("Private key:", private)

    # Encrypt and decrypt a message
    message = " This is Secret!"
    encrypted_msg = encrypt(public, message)
    print("Encrypted message:", encrypted_msg)

    decrypted_msg = decrypt(private, encrypted_msg)
    print("Decrypted message:", decrypted_msg)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 21.6 MB/s eta 0:00:00
Generating RSA keypair...
Generating large safe primes...
factorization  of p  69175439524525058761851597998808434539904586788865981955967350794143576087237081622997197239913106241690604463807830343751887093671381302510118771863355293937412642500462759942330773361772153088993219436520952934522808984468052330095014152433880794613397498247688494741106220453674172197006950915279652566621245377267207729279619290091969677049494211819581219283414573287257599984190558288789063608479443102227514084789923316772211567672037145003494256861547831618961364125121273353391002022815502943
factorization  of q  8794815025442367175872317690184043049960913958926027401246014551323037487279003878115962822991026177834008920992607937715880810392569525478006247877553299205514059203812904975515072622754095232170106970490312073109673754363943135291471431596904270487770986215321374180539375894352696544328253351314413740889798134196919413

In [ ]:
def count_digits(n):
    return len(str(n))

# Example
num =   69175439524525058761851597998808434539904586788865981955967350794143576087237081622997197239913106241690604463807830343751887093671381302510118771863355293937412642500462759942330773361772153088993219436520952934522808984468052330095014152433880794613397498247688494741106220453674172197006950915279652566621245377267207729279619290091969677049494211819581219283414573287257599984190558288789063608479443102227514084789923316772211567672037145003494256861547831618961364125121273353391002022815502943
print(count_digits(num))  # Output: 9


In [5]:
#!pip install gmpy2
#!pip install pycryptodome
#!pip install Crypto
import gmpy2
from Crypto.Util.number import getPrime, inverse

def generate_safe_prime(digits):
    """Generates a safe prime with at least 'digits' decimal digits."""
    bits = digits * 3.321928  # Convert decimal digits to bits
    bits = int(bits) + 1  # Ensure rounding up
    while True:
        p = getPrime(bits)  # Fast prime generation
        if gmpy2.is_prime((p - 1) // 2):  # Ensure safe prime condition
            return p

def generate_keypair(digits=400):
    """Generates RSA keypair with at least 'digits' in modulus."""
    print("Generating large safe primes...")
    p, q = generate_safe_prime(digits // 2), generate_safe_prime(digits // 2)

    n = p * q
    while len(str(n)) < digits:  # Ensure n has at least 400 digits
        p, q = generate_safe_prime(digits // 2), generate_safe_prime(digits // 2)
        n = p * q

    phi = (p - 1) * (q - 1)

    e = 65537  # Standard public exponent (fast, secure)
    d = inverse(e, phi)
    print("factorization  of p " , p)
    print("factorization  of q " , q)
    return (e, n), (d, n)

def encrypt(pk, plaintext):
    """Encrypt a message using RSA."""
    e, n = pk
    return [pow(ord(char), e, n) for char in plaintext]

def decrypt(pk, ciphertext):
    """Decrypt a message using RSA."""
    d, n = pk
    return ''.join(chr(pow(char, d, n)) for char in ciphertext)

# --- Main Program ---
if __name__ == '__main__':
    # Generate RSA keypair with at least 1200-digit modulus
    print("Generating RSA keypair...")
    public, private = generate_keypair(1200)
    print("Keypair generated.")

    print("Public key:", public)
    print("Private key:", private)

    # Encrypt and decrypt a message
    message = " This is Secret!"
    encrypted_msg = encrypt(public, message)
    print("Encrypted message:", encrypted_msg)

    decrypted_msg = decrypt(private, encrypted_msg)
    print("Decrypted message:", decrypted_msg)


Generating RSA keypair...
Generating large safe primes...
factorization  of p  1686845564582776514604457641547825698195582319765128446272113898831910579156562788059041896242097230299202643715662205255575378785631313428396450041423024278966379483561289075001997248295808145807232976052241670896924717491338179525955139293576890712741518554920276906410854459249412907984640601486258008585604200586615755082890277183991656082571675674417018003507527742376602387877773644041366936837572524526305622839044891361368082611967573101287125997193881965488741828410565769959259031880472103638204329538386663678387277293717489115781719870071719945053805146435937499398019036278287775413110563
factorization  of q  1045443103510183274119011959357567105696777608157519834236085996261742181534607545131095032097641530926067435777141933520976940521396236269524255556885731877721719158840869477995567360969249804846045263471962956143438030800985619082218010926654925162017359819209052944555741270016710170549881630934

n this RSA demonstration, I wanted a 1200-digit modulus. To do that, I generated safe primes that were roughly 600 digits each, for p and q. Generating safe primes adds some security. Then, the modulus was factored in order to find the required primes so the private key could be computed. I chose a public exponent 'e', encrypted a message with the public key, and demonstrated successful decryption with the matching private key.

In [6]:
def count_digits(n):
    return len(str(n))

# Example
num =    1686845564582776514604457641547825698195582319765128446272113898831910579156562788059041896242097230299202643715662205255575378785631313428396450041423024278966379483561289075001997248295808145807232976052241670896924717491338179525955139293576890712741518554920276906410854459249412907984640601486258008585604200586615755082890277183991656082571675674417018003507527742376602387877773644041366936837572524526305622839044891361368082611967573101287125997193881965488741828410565769959259031880472103638204329538386663678387277293717489115781719870071719945053805146435937499398019036278287775413110563
print(count_digits(num))  # Output: 9


601
